# HR Employee Handbook RAG System (v2)

This notebook:
1. Loads `employee_handbook_print_1.pdf`
2. Splits each page into **sections by heading**, then chunks within each section
3. Creates OpenAI embeddings for each chunk
4. Stores them in the Supabase table `hr_documents` (clearing old rows first)
5. Demonstrates semantic search with a **keyword-boosted rerank** step

### What changed from v1

| Problem in v1 | Fix in v2 |
|---|---|
| Fixed-size 500/50 chunking sliced headings away from their own body text (e.g. `TERMINATION OF EMPLOYMENT` ended one chunk, the policy text started the next) | Chunk **within sections** — split on ALL-CAPS headings first, then prepend the heading to every chunk under it |
| 500-char chunks were too tight for policy paragraphs, frequently cutting mid-clause | Chunk size raised to 900 chars, overlap raised to 150 (~17%) |
| Pure cosine similarity couldn't separate "employee termination" from "student dismissal" (top score was only 0.597) | Retrieve a wider candidate pool, then rerank with a cheap keyword-overlap boost |

**Note:** because the chunk boundaries are different from v1, this notebook clears `hr_documents` before re-embedding. Don't run this against a table you need to keep v1 data in.

## 0. Prerequisites

### Supabase SQL — run once in your project's SQL editor (same schema as v1, no change needed if you already ran this)

```sql
-- Enable the pgvector extension (if not already enabled)
create extension if not exists vector;

-- Table for the HR handbook
create table if not exists hr_documents (
  id          bigserial primary key,
  content     text        not null,
  page_number int,
  chunk_index int,
  embedding   vector(1536)
);

-- Similarity-search function for this table
create or replace function match_hr_documents (
  query_embedding vector(1536),
  match_threshold float,
  match_count     int
)
returns table (
  id          bigint,
  content     text,
  page_number int,
  chunk_index int,
  similarity  float
)
language sql stable
as $$
  select
    id,
    content,
    page_number,
    chunk_index,
    1 - (embedding <=> query_embedding) as similarity
  from hr_documents
  where 1 - (embedding <=> query_embedding) > match_threshold
  order by embedding <=> query_embedding
  limit match_count;
$$;
```

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client
from openai import OpenAI
from pypdf import PdfReader   # pip install pypdf

load_dotenv(override=True)

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
SUPABASE_URL     = "https://ssrxdvbnjfruzikvages.supabase.co"
SUPABASE_API_KEY = os.getenv("SUBABASE_API_KEY")   # matches original .env key name
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")

PDF_PATH         = Path("raz/employee_handbook_print_1.pdf")  # adjust path if needed
TABLE_NAME       = "hr_documents"
EMBED_MODEL      = "text-embedding-3-small"

CHUNK_SIZE       = 900   # characters per chunk (was 500 in v1)
CHUNK_OVERLAP    = 150   # overlap between consecutive chunks (was 50 in v1)

# A heading line in this handbook is a short, ALL-CAPS line (e.g. "TERMINATION OF EMPLOYMENT").
# Tune this regex if your PDF uses a different heading style.
HEADING_RE = re.compile(r"^[A-Z][A-Z0-9 ,&\-/]{3,80}$")

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH.resolve()}"

In [ ]:
# ── Clients ──────────────────────────────────────────────────────────────────
supabase = create_client(SUPABASE_URL, SUPABASE_API_KEY)
openai   = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# ── Step 1: Extract text from PDF ────────────────────────────────────────────
def extract_pages(pdf_path: Path) -> list[dict]:
    """Return a list of {page_number, text} dicts for every page in the PDF."""
    pages = []
    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append({"page_number": i, "text": text})
    print(f"Extracted text from {len(pages)} pages.")
    return pages

pages = extract_pages(PDF_PATH)

## Step 2: Heading-aware chunking

Instead of slicing each page into raw fixed-size windows, we:

1. Walk the page text line by line and group it under its most recent ALL-CAPS heading
2. Chunk **within** each section (so a chunk never straddles two unrelated policies)
3. Prepend the heading to every chunk's embedded content, so the chunk is self-contained even without overlap rescuing it

If a section has no heading above it yet (e.g. the very top of a page), `heading` is `None` and we just chunk the body on its own.

In [ ]:

# ── Worked example: how a heading gets attached to each chunk ───────────────
#
# Page text:
#   "TERMINATION OF EMPLOYMENT
#    Notice must be given two weeks in advance. ... (long section) ..."
#
# Step 1 — split_by_headings() pairs the heading with its full body text:
#   ("TERMINATION OF EMPLOYMENT", "Notice must be given two weeks...")
#
# Step 2 — chunk_text() only chops the BODY (never the heading itself).
#   If the section is long, this can produce multiple chunks:
#     chunk0 = "Notice must be given two weeks in advance..."
#     chunk1 = "...continuation of the same section..."
#     chunk2 = "...rest of the section..."
#
# Step 3 — the SAME heading is prepended to EVERY chunk from that section:
#     content = f"{heading}\n\n{chunk}" if heading else chunk
#
#   Result:
#     chunk0 -> "TERMINATION OF EMPLOYMENT\n\nNotice must be given two weeks..."
#     chunk1 -> "TERMINATION OF EMPLOYMENT\n\n...continuation of the same section..."
#     chunk2 -> "TERMINATION OF EMPLOYMENT\n\n...rest of the section..."
#
# Why repeat the heading on every chunk instead of just once?
#   We don't know in advance WHICH chunk will be the closest match to a
#   future query — so every chunk needs to be self-contained and carry the
#   section title with it, even if the heading itself ends up "far away"
#   from that particular slice of text in the original PDF.
#
# Edge case: if a page opens with body text before any heading appears,
# split_by_headings() returns heading=None for that segment, and the
# "if heading else chunk" check just skips the prepend — nothing to attach.
# ───────────────────────────────────────────────────────────────────────────

def split_by_headings(text: str) -> list[tuple[str | None, str]]:
    """Walk page text, grouping it under its most recent ALL-CAPS heading.

    Returns a list of (heading_or_None, body_text) tuples in reading order.
    """
    segments: list[tuple[str | None, str]] = []
    buffer: list[str] = []
    current_heading: str | None = None

    for line in text.split("\n"):
        stripped = line.strip()
        if HEADING_RE.match(stripped):
            if buffer:
                segments.append((current_heading, "\n".join(buffer).strip()))
                buffer = []
            current_heading = stripped
        else:
            buffer.append(line)

    if buffer:
        segments.append((current_heading, "\n".join(buffer).strip()))

    return [(h, b) for h, b in segments if b]   # drop empty bodies

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping fixed-size character chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]   # drop empty strings

In [ ]:
# ── Worked example: chunk_text() no longer cuts words in half ────────────────
#
# Text:   "Notice must be given to Termination"
#          Index:  0         1         2
#                  0123456789012345678901234567890123456
#
# chunk_size = 20, overlap = 5, start = 0
#
# Step 1 — guess the ideal cut point:
#   end = min(start + chunk_size, n) = min(0 + 20, 37) = 20
#   text[20] = "T"   <- lands INSIDE the word "Termination"
#
# Step 2 — back up one character at a time until we land on a space:
#   end=20 -> "T" (not space) -> end -= 1
#   end=19 -> " " (space!)    -> stop
#
# Step 3 — slice using the corrected end:
#   chunk = text[0:19] = "Notice must be given to"
#   (a clean, complete chunk — no broken word at the end)
#
# Without the backward-search step, the old chunk_text() would have cut at
# the raw index 20 and produced "...given to T", with "ermination" pushed
# into the next chunk — the same bug that produced "nation of employment"
# in the real handbook output.
# ───────────────────────────────────────────────────────────────────────────



def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks, breaking on whitespace so words
    are never cut in half."""
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        # back off to the nearest space so we don't split a word
        if end < n:
            while end > start and not text[end].isspace():
                end -= 1
            if end == start:          # no space found, fall back to hard cut
                end = min(start + chunk_size, n)
        chunks.append(text[start:end].strip())
        start = end - overlap if end - overlap > start else end
    return [c for c in chunks if c]

In [ ]:

# Build flat list of {content, page_number, chunk_index}
all_chunks = []
for page in pages:
    for heading, body in split_by_headings(page["text"]):
        for idx, chunk in enumerate(chunk_text(body)):
            content = f"{heading}\n\n{chunk}" if heading else chunk
            all_chunks.append({
                "content":     content,
                "page_number": page["page_number"],
                "chunk_index": idx,
            })

print(f"Total chunks to embed: {len(all_chunks)}")
print("\nSample chunk:\n", all_chunks[0]["content"][:300] if all_chunks else "(none)")

## Step 3: Clear out old chunks

The chunk boundaries above don't match the v1 fixed-size scheme, so we wipe `hr_documents` before re-uploading. **Skip this cell if you want to keep existing rows for some other reason.**

In [ ]:
# ── Clear existing rows before re-embedding with the new chunking scheme ─────
deleted = supabase.table(TABLE_NAME).delete().neq("id", 0).execute()
print(f"Cleared existing rows from {TABLE_NAME}.")

In [ ]:
# ── Step 4: Embed & upload ────────────────────────────────────────────────────
def get_embedding(text: str) -> list[float]:
    """Return the embedding vector for a single piece of text."""
    response = openai.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding


def upload_chunks(chunks: list[dict], batch_size: int = 20) -> None:
    """
    Embed each chunk and upsert into Supabase in batches.
    A small sleep between batches avoids OpenAI rate-limit errors.
    """
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i : i + batch_size]
        rows  = []
        for chunk in batch:
            embedding = get_embedding(chunk["content"])
            rows.append({
                "content":     chunk["content"],
                "page_number": chunk["page_number"],
                "chunk_index": chunk["chunk_index"],
                "embedding":   embedding,
            })

        supabase.table(TABLE_NAME).insert(rows).execute()
        print(f"  Uploaded chunks {i+1}\u2013{min(i+batch_size, total)} of {total}")
        time.sleep(0.5)   # be kind to the OpenAI rate limiter


print("Starting embedding + upload \u2026")
upload_chunks(all_chunks)
print("Done \u2014 all chunks stored in", TABLE_NAME)

## Step 5: Semantic search with keyword-boosted rerank

Plain cosine similarity struggled to tell "employee termination" apart from "student dismissal" \u2014 the top score in v1 topped out around 0.597. To fix this without adding a new dependency:

1. Pull a **wider candidate pool** from pgvector (`candidate_k`, default 15) at a lower threshold
2. Re-score each candidate by adding a small bonus for literal word overlap with the query
3. Re-sort and return only the top `top_k`

This nudges chunks that actually contain the query's words above chunks that are merely topically nearby in embedding space.

In [ ]:
WORD_RE = re.compile(r"[a-z]+")

def search_handbook(query: str, threshold: float = 0.3, top_k: int = 5,
                    candidate_k: int = 15, keyword_weight: float = 0.05) -> list[dict]:
    """Return the most relevant HR handbook passages for a query.

    Retrieves `candidate_k` results by vector similarity, then reranks by
    adding `keyword_weight` per literal query-word match found in the chunk.
    """
    query_embedding = get_embedding(query)
    result = supabase.rpc("match_hr_documents", {
        "query_embedding": query_embedding,
        "match_threshold":  threshold,
        "match_count":      candidate_k,
    }).execute()
    candidates = result.data

    query_terms = set(WORD_RE.findall(query.lower()))
    for c in candidates:
        text_terms = set(WORD_RE.findall(c["content"].lower()))
        c["keyword_score"]  = len(query_terms & text_terms)
        c["combined_score"] = c["similarity"] + keyword_weight * c["keyword_score"]

    candidates.sort(key=lambda c: c["combined_score"], reverse=True)
    return candidates[:top_k]

In [ ]:
# Query 1 \u2014 Board meeting public participation
results = search_handbook("How do I address the board of education?")
print("Query: How do I address the board of education?")
for r in results:
    print(f"  [Page {r['page_number']}, chunk {r['chunk_index']},"
          f" sim={r['similarity']:.3f}, kw={r['keyword_score']}, combined={r['combined_score']:.3f}]")
    print("  ", r["content"][:200], "\u2026\n")

In [ ]:
# Query 2 \u2014 Payroll deductions
results = search_handbook("what is the sick vacation?")
print("Query: Can I stop payroll deductions for my union?")
for r in results:
    print(f"  [Page {r['page_number']}, chunk {r['chunk_index']},"
          f" sim={r['similarity']:.3f}, kw={r['keyword_score']}, combined={r['combined_score']:.3f}]")
    print("  ", r["content"][:200], "\u2026\n")

In [ ]:
# Query 3 \u2014 Termination policy (the query that exposed the v1 chunking problem)
results = search_handbook("what is termination policy?")
print("Query: what is termination policy?")
for r in results:
    print(f"  [Page {r['page_number']}, chunk {r['chunk_index']},"
          f" sim={r['similarity']:.3f}, kw={r['keyword_score']}, combined={r['combined_score']:.3f}]")
    print("  ", r["content"][:200], "\u2026\n")